# 99. Конфигурация Excel Loader

Этот ноутбук помогает:
1. **Обновить Excel шаблон** (добавить все метрики из БД)
2. **Создать упрощенный шаблон** для Standard/Lite моделей (только канонические метрики)
3. **Экспортировать данные из БД в Excel шаблон** (для ручной корректировки)
4. **Загрузить данные из Excel в БД** (после корректировки)
5. Проверить структуру Excel-файла и увидеть алиасы метрик
6. Сгенерировать YAML-фрагменты для `configs/excel_loader.yaml`

**Доступные шаблоны:**
- `template_UNIFIED_All_Data.xlsx` - полный шаблон для всех моделей (Standard, Custom, Lite) - содержит все метрики из БД
- `template_STANDARD_LITE.xlsx` - упрощенный шаблон только для Standard и Lite моделей - содержит только канонические метрики

**Рабочий процесс:**
1. Выбор шаблона (полный или упрощенный)
2. Экспорт данных из БД → Excel (если данные уже в БД)
3. Ручная корректировка данных в Excel
4. Загрузка данных из Excel → БД (с поддержкой тестовой БД)
5. Проверка и валидация загруженных данных



In [ ]:
from pathlib import Path
from collections import Counter
import sys
import subprocess

import pandas as pd
import yaml

# Автоматическое определение компании из пути ноутбука
current_dir = Path.cwd()
if 'companies' in current_dir.parts:
    companies_idx = current_dir.parts.index('companies')
    if companies_idx + 1 < len(current_dir.parts):
        COMPANY = current_dir.parts[companies_idx + 1]
    else:
        COMPANY = 'rusal'  # fallback
else:
    COMPANY = 'rusal'  # fallback

# Определение корня проекта
if (current_dir / 'engine').exists():
    PROJECT_ROOT = current_dir
elif (current_dir.parent / 'engine').exists():
    PROJECT_ROOT = current_dir.parent
elif (current_dir.parent.parent / 'engine').exists():
    PROJECT_ROOT = current_dir.parent.parent
else:
    PROJECT_ROOT = current_dir.parent.parent.parent

sys.path.insert(0, str(PROJECT_ROOT))

# Настройки путей
COMPANY_ROOT = PROJECT_ROOT / 'companies' / COMPANY
EXCEL_INPUT = COMPANY_ROOT / "data" / "excel" / f"{COMPANY}_input.xlsx"
EXCEL_FILLED = COMPANY_ROOT / "data" / "excel" / f"{COMPANY}_filled.xlsx"
CONFIG_PATH = COMPANY_ROOT / "configs" / "excel_loader.yaml"
TEMPLATE_PATH = PROJECT_ROOT / "templates" / "excel_templates" / "template_UNIFIED_All_Data.xlsx"

print(f"📁 Компания: {COMPANY}")
print(f"📁 Корень проекта: {PROJECT_ROOT}")
print(f"📁 Excel input: {EXCEL_INPUT}")
print(f"📁 Excel filled: {EXCEL_FILLED}")
print(f"📁 Config: {CONFIG_PATH}")
print(f"📁 Template: {TEMPLATE_PATH}")


In [ ]:
# Обновление Excel шаблона: добавление всех метрик из БД
# ВАЖНО: Это обновит сам шаблон, добавив все метрики из БД

if TEMPLATE_PATH.exists():
    update_cmd = [
        "python3",
        str(PROJECT_ROOT / "tools" / "update_excel_template_metrics.py"),
        "--company", COMPANY,
        "--template", str(TEMPLATE_PATH)
    ]
    
    print("🔧 Обновление Excel шаблона...")
    print(f"Команда: {' '.join(update_cmd)}")
    print(f"📄 Шаблон: {TEMPLATE_PATH}")
    print("\n⚠️ Для реального обновления раскомментируйте код ниже:")
    
    # Раскомментируйте для реального обновления:
    # env = os.environ.copy()
    # env['PYTHONPATH'] = str(PROJECT_ROOT) + ':' + env.get('PYTHONPATH', '')
    # result = subprocess.run(update_cmd, cwd=PROJECT_ROOT, env=env, capture_output=True, text=True)
    # print(result.stdout)
    # if result.returncode != 0:
    #     print("❌ Ошибка при обновлении:")
    #     print(result.stderr)
    # else:
    #     print("✅ Шаблон успешно обновлен!")
else:
    print(f"⚠️ Шаблон не найден: {TEMPLATE_PATH}")


## 1️⃣ Экспорт данных из БД в Excel шаблон

Если данные уже загружены в БД (например, из EDGAR), можно экспортировать их в Excel для ручной корректировки.


### 0.2 Создание упрощенного шаблона для Standard/Lite моделей

Создает упрощенный шаблон, содержащий только канонические метрики, необходимые для Standard и Lite моделей.


In [ ]:
# Создание упрощенного шаблона для Standard/Lite моделей
# Этот шаблон содержит только канонические метрики

STANDARD_TEMPLATE_PATH = PROJECT_ROOT / "templates" / "excel_templates" / "template_STANDARD_LITE.xlsx"

if TEMPLATE_PATH.exists():
    create_standard_cmd = [
        "python3",
        str(PROJECT_ROOT / "tools" / "create_standard_template.py"),
        "--template", str(TEMPLATE_PATH),
        "--output", str(STANDARD_TEMPLATE_PATH)
    ]
    
    print("🔧 Создание упрощенного шаблона для Standard/Lite...")
    print(f"Команда: {' '.join(create_standard_cmd)}")
    print(f"📄 Исходный шаблон: {TEMPLATE_PATH}")
    print(f"📄 Выходной шаблон: {STANDARD_TEMPLATE_PATH}")
    print("\n⚠️ Для реального создания раскомментируйте код ниже:")
    
    # Раскомментируйте для реального создания:
    # env = os.environ.copy()
    # env['PYTHONPATH'] = str(PROJECT_ROOT) + ':' + env.get('PYTHONPATH', '')
    # result = subprocess.run(create_standard_cmd, cwd=PROJECT_ROOT, env=env, capture_output=True, text=True)
    # print(result.stdout)
    # if result.returncode != 0:
    #     print("❌ Ошибка при создании:")
    #     print(result.stderr)
    # else:
    #     print("✅ Упрощенный шаблон успешно создан!")
    #     print(f"   Используйте его для Standard и Lite моделей")
else:
    print(f"⚠️ Исходный шаблон не найден: {TEMPLATE_PATH}")


In [ ]:
# Экспорт данных из БД в Excel шаблон
import os

if TEMPLATE_PATH.exists():
    export_cmd = [
        "python3", 
        str(PROJECT_ROOT / "tools" / "export_data_to_excel_template.py"),
        "--company", COMPANY,
        "--output", str(EXCEL_FILLED),
        "--template", str(TEMPLATE_PATH)
    ]
    
    print("🚀 Запуск экспорта данных из БД в Excel...")
    print(f"Команда: {' '.join(export_cmd)}")
    
    env = os.environ.copy()
    env['PYTHONPATH'] = str(PROJECT_ROOT) + ':' + env.get('PYTHONPATH', '')
    
    result = subprocess.run(export_cmd, cwd=PROJECT_ROOT, env=env, capture_output=True, text=True)
    
    if result.returncode == 0:
        print("✅ Экспорт выполнен успешно!")
        if EXCEL_FILLED.exists():
            print(f"📄 Файл создан: {EXCEL_FILLED}")
            print(f"   Размер: {EXCEL_FILLED.stat().st_size / 1024:.1f} KB")
    else:
        print("❌ Ошибка при экспорте:")
        print(result.stderr)
else:
    print(f"⚠️ Шаблон не найден: {TEMPLATE_PATH}")


## 2️⃣ Загрузка данных из Excel в БД

После ручной корректировки данных в Excel, загрузите их обратно в БД.


In [ ]:
# Загрузка данных из Excel в БД
# ВАЖНО: Используйте EXCEL_FILLED или EXCEL_INPUT в зависимости от того, какой файл вы редактировали

EXCEL_TO_LOAD = EXCEL_FILLED if EXCEL_FILLED.exists() else EXCEL_INPUT

# 🧪 ТЕСТОВАЯ БД: Для тестирования используем отдельную БД, чтобы не повредить основную
USE_TEST_DB = True  # Установите False для загрузки в основную БД
TEST_DB_PATH = COMPANY_ROOT / "data_mart_test.db"

if EXCEL_TO_LOAD.exists():
    load_cmd = [
        "python3",
        str(PROJECT_ROOT / "tools" / "load_excel_to_data_mart.py"),
        "--company", COMPANY,
        "--file", str(EXCEL_TO_LOAD),
        "--config", str(CONFIG_PATH) if CONFIG_PATH.exists() else str(PROJECT_ROOT / "configs" / "excel_loader.yaml")
    ]
    
    # Добавляем путь к тестовой БД, если включен режим тестирования
    if USE_TEST_DB:
        load_cmd.extend(["--test-db-path", str(TEST_DB_PATH)])
        print("🧪 РЕЖИМ ТЕСТИРОВАНИЯ: данные будут загружены в тестовую БД")
        print(f"   Тестовая БД: {TEST_DB_PATH}")
        print("   ⚠️ Основная БД не будет затронута")
    else:
        print("⚠️ РЕЖИМ ПРОДАКШЕН: данные будут загружены в основную БД")
        print("   Убедитесь, что вы хотите перезаписать данные!")
    
    print(f"\n🚀 Команда загрузки:")
    print(f"   {' '.join(load_cmd)}")
    print(f"📄 Файл: {EXCEL_TO_LOAD}")
    print("\n⚠️ Для реальной загрузки раскомментируйте код ниже:")
    
    # Раскомментируйте для реальной загрузки:
    # env = os.environ.copy()
    # env['PYTHONPATH'] = str(PROJECT_ROOT) + ':' + env.get('PYTHONPATH', '')
    # result = subprocess.run(load_cmd, cwd=PROJECT_ROOT, env=env, capture_output=True, text=True)
    # if result.returncode == 0:
    #     print("✅ Загрузка выполнена успешно!")
    #     if USE_TEST_DB:
    #         print(f"📊 Данные загружены в тестовую БД: {TEST_DB_PATH}")
    #     else:
    #         print("📊 Данные загружены в основную БД")
    # else:
    #     print("❌ Ошибка при загрузке:")
    #     print(result.stderr)
else:
    print(f"⚠️ Excel файл не найден: {EXCEL_TO_LOAD}")


## 3️⃣ Проверка структуры Excel файла

Проверяем структуру Excel файла и анализируем метрики.


In [ ]:
# Загружаем Excel файл для анализа
EXCEL_TO_ANALYZE = EXCEL_FILLED if EXCEL_FILLED.exists() else EXCEL_INPUT

if EXCEL_TO_ANALYZE.exists():
    book = pd.read_excel(EXCEL_TO_ANALYZE, sheet_name=None, dtype=object)
    print(f"📊 Найдено листов: {len(book)}")
    for name, df in book.items():
        print(f"   - {name:<30} rows={len(df):<5} cols={len(df.columns)}")
else:
    print(f"⚠️ Excel файл не найден: {EXCEL_TO_ANALYZE}")
    book = {}


## 4️⃣ Сравнение данных между основной и тестовой БД

После загрузки данных в тестовую БД, можно сравнить их с основной БД и запустить препроцесс для проверки.


In [ ]:
# Сравнение данных между основной и тестовой БД
# ВАЖНО: Убедитесь, что данные загружены в тестовую БД перед сравнением

if TEST_DB_PATH.exists():
    compare_cmd = [
        "python3",
        str(PROJECT_ROOT / "tools" / "compare_test_db.py"),
        "--company", COMPANY,
        "--test-db", str(TEST_DB_PATH),
        "--run-preprocess"  # Запустить препроцесс на тестовой БД
    ]
    
    print("🔍 Запуск сравнения данных и препроцесса...")
    print(f"Команда: {' '.join(compare_cmd)}")
    print(f"🧪 Тестовая БД: {TEST_DB_PATH}")
    print(f"📊 Основная БД: {PROJECT_ROOT / 'data_mart.db'}")
    print("\n⚠️ Для реального запуска раскомментируйте код ниже:")
    
    # Раскомментируйте для реального запуска:
    # env = os.environ.copy()
    # env['PYTHONPATH'] = str(PROJECT_ROOT) + ':' + env.get('PYTHONPATH', '')
    # result = subprocess.run(compare_cmd, cwd=PROJECT_ROOT, env=env, capture_output=True, text=True)
    # print(result.stdout)
    # if result.returncode != 0:
    #     print("❌ Ошибка при сравнении:")
    #     print(result.stderr)
else:
    print(f"⚠️ Тестовая БД не найдена: {TEST_DB_PATH}")
    print("   Сначала загрузите данные в тестовую БД (ячейка выше)")


In [ ]:
summary = []
for sheet_name, column in [
    ("history_IS", "metric"),
    ("history_BS", "metric"),
    ("history_CF", "metric"),
    ("segments_financial", "metric"),
    ("segments_operational", "metric"),
    ("operational_drivers", "driver_name"),
]:
    if sheet_name not in book:
        continue
    series = book[sheet_name][column].dropna().astype(str).str.strip()
    counts = Counter(series)
    summary.append({
        "sheet": sheet_name,
        "unique": len(counts),
        "top5": counts.most_common(5),
    })

pd.DataFrame(summary)


In [ ]:
if CONFIG_PATH.exists():
    loader_cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
else:
    loader_cfg = {}
loader_cfg.keys()


In [ ]:
def suggest_aliases(sheet: str, column: str = "metric") -> pd.DataFrame:
    if sheet not in book:
        return pd.DataFrame()
    series = book[sheet][column].dropna().astype(str).str.strip()
    counts = Counter(series)
    data = []
    for metric, count in counts.most_common():
        alias = metric.lower().replace(" ", "_")
        data.append({"sheet": sheet, "metric_raw": metric, "alias": alias, "count": count})
    return pd.DataFrame(data)

suggest_aliases("history_IS").head(10)


In [ ]:
aliases_df = pd.concat([
    suggest_aliases("history_IS"),
    suggest_aliases("history_BS"),
    suggest_aliases("history_CF"),
], ignore_index=True)

# оставляем только новые алиасы, которых нет в словаре
existing_aliases = set()
metrics_dict = loader_cfg.get("validation", {}).get("dictionaries", {}).get("metrics", {})
if metrics_dict:
    sheet_name = metrics_dict.get("sheet")
    aliases_column = metrics_dict.get("aliases_column", "accepted_aliases")
    key_column = metrics_dict.get("key_column", "canonical_metric")
    if sheet_name in book:
        df_dict = book[sheet_name]
        for _, row in df_dict.iterrows():
            canonical = str(row.get(key_column)).strip().lower()
            aliases_raw = str(row.get(aliases_column) or "")
            for alias in aliases_raw.split(";"):
                alias_norm = alias.strip().lower()
                if alias_norm:
                    existing_aliases.add(alias_norm)

new_aliases = aliases_df[~aliases_df["alias"].isin(existing_aliases)]
new_aliases.head(20)


In [ ]:
def build_yaml_alias_block(df: pd.DataFrame, statement: str) -> str:
    lines = [f"  {statement.upper()}:" ]
    for _, row in df.sort_values("metric_raw").iterrows():
        alias = row["alias"].strip()
        canonical = alias
        lines.append(f"    - source_metric: {alias}")
        lines.append(f"      target_metric: {canonical}")
        lines.append(f"      scale: auto")
        lines.append(f"      sign: auto")
    return "\n".join(lines)

build_yaml_alias_block(new_aliases[new_aliases["sheet"] == "history_IS"], "is")


## Итог
- Скопируйте сгенерированный YAML-блок в `configs/excel_loader.yaml` (или company-specific override).
- Запустите `python tools/load_excel_to_data_mart.py --dry-run`.
- После успешной загрузки обновите документацию и синхронизируйте release.

